# LD Clumping

This notebook performs LD-based clumping on the HF prioritized variant list to identify independent association signals across 52 macrophage-associated candidate genes. Clumping is run per chromosome using PLINK 2.0 against a 1000 Genomes EUR reference panel (GRCh38, r² = 0.1, 1 Mb window), producing a reduced set of independent lead SNPs for downstream interpretation.

In [10]:
from pathlib import Path
import pandas as pd
import subprocess

In [11]:
hf_csv_path   = Path("../data/final/hf_prioritized_variants.csv")
ref_dir       = Path("../data/raw/1000genomes")
sumstats_dir  = Path("../data/raw/1000genomes/sumstats")
clump_raw_dir = Path("../data/interim/ld_clumping")
out_dir       = Path("../data/final/ld_clumping")

for d in [sumstats_dir, clump_raw_dir, out_dir]:
    d.mkdir(parents=True, exist_ok=True)

In [12]:
R2_THRESHOLD  = 0.1
KB_WINDOW     = 1000    # 1 Mb window
REFERENCE_POP = "1000 Genomes EUR (CEU+TSI+FIN+GBR+IBS) GRCh38 high-coverage"

hf_df = pd.read_csv(hf_csv_path, sep=";", low_memory=False)

In [13]:
# Run separately per dataset as professor requested
datasets = sorted(hf_df["source_dataset"].unique().tolist())
print(f"Datasets: {datasets}")
print()

dataset_inputs = {}

for dataset in datasets:
    ds_df = hf_df[hf_df["source_dataset"] == dataset].copy()

    clump_input_ds = (
        ds_df[
            ds_df["rsID"].notna() &
            ds_df["chromosome"].notna() &
            ds_df["p_value"].notna()
        ]
        .sort_values("p_value")
        .drop_duplicates(subset=["rsID"])
        .copy()
    )
    clump_input_ds["chromosome"] = clump_input_ds["chromosome"].astype(int)

    chromosomes_ds = sorted(clump_input_ds["chromosome"].unique().tolist())
    dataset_inputs[dataset] = {
        "df":          clump_input_ds,
        "chromosomes": chromosomes_ds,
    }

    print(f"{dataset}: {len(clump_input_ds):,} variants | chromosomes: {chromosomes_ds}")

    # Write per-chromosome sumstats files for this dataset
    ds_tag = dataset.lower().replace(" ", "_")
    for chrom in chromosomes_ds:
        chrom_df = (
            clump_input_ds[clump_input_ds["chromosome"] == chrom]
            [["rsID", "p_value"]]
            .rename(columns={"rsID": "SNP", "p_value": "P"})
        )
        out_path = sumstats_dir / f"hf_{ds_tag}_chr{chrom}_sumstats.txt"
        chrom_df.to_csv(out_path, sep="\t", index=False)

print("\nSummary statistics files written.")

Datasets: ['FinnGen', 'GWAS Catalog', 'HERMES']

FinnGen: 47,050 variants | chromosomes: [1, 2, 4, 5, 6, 7, 8, 9, 11, 12, 13, 15, 16, 17, 19, 20, 21]
GWAS Catalog: 0 variants | chromosomes: []
HERMES: 19,942 variants | chromosomes: [1, 2, 4, 5, 6, 7, 8, 9, 11, 12, 13, 15, 16, 17, 19, 20, 21]

Summary statistics files written.


In [14]:
def run_plink_clump(chrom, ds_tag, ref_dir, sumstats_dir, out_dir, r2, kb):
    pgen_file     = ref_dir / f"chr{chrom}_hg38.pgen"
    pvar_file     = ref_dir / f"chr{chrom}_hg38_rs.pvar.zst"
    psam_file     = ref_dir / "hg38_corrected.psam"
    sumstats_file = sumstats_dir / f"hf_{ds_tag}_chr{chrom}_sumstats.txt"
    out_prefix    = out_dir / f"clumped_{ds_tag}_chr{chrom}"

    for f in [pgen_file, pvar_file, psam_file, sumstats_file]:
        if not f.exists():
            print(f"  SKIP: {f.name} not found")
            return None, None

    cmd = [
        "plink2",
        "--pgen",           str(pgen_file),
        "--pvar",           str(pvar_file),
        "--psam",           str(psam_file),
        "--keep-if",        "SuperPop = EUR",
        "--rm-dup",         "exclude-all",
        "--clump",          str(sumstats_file),
        "--clump-id-field", "SNP",
        "--clump-p-field",  "P",
        "--clump-p1",       "1",
        "--clump-p2",       "1",
        "--clump-r2",       str(r2),
        "--clump-kb",       str(kb),
        "--out",            str(out_prefix),
        "--silent",
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"  ERROR on {ds_tag} chr{chrom}:")
        for line in result.stderr.strip().split("\n")[-10:]:
            print(f"    {line}")
        return None, None

    clumps_file  = Path(str(out_prefix) + ".clumps")
    missing_file = Path(str(out_prefix) + ".clumps.missing_id")

    n_clumps  = 0
    n_missing = 0

    if clumps_file.exists():
        n_clumps = len(pd.read_csv(clumps_file, sep=r"\s+"))

    if missing_file.exists():
        with open(missing_file) as f:
            n_missing = sum(1 for line in f) - 1

    return n_clumps, n_missing

clump_summary = []

for dataset, info in dataset_inputs.items():
    ds_tag = dataset.lower().replace(" ", "_")
    print(f"\n── {dataset} ──")

    for chrom in info["chromosomes"]:
        print(f"  Chr{chrom:>2}: clumping...", end=" ", flush=True)

        n_clumps, n_missing = run_plink_clump(
            chrom        = chrom,
            ds_tag       = ds_tag,
            ref_dir      = ref_dir,
            sumstats_dir = sumstats_dir,
            out_dir      = clump_raw_dir,
            r2           = R2_THRESHOLD,
            kb           = KB_WINDOW,
        )

        if n_clumps is not None:
            print(f"{n_clumps} lead SNPs | {n_missing} not in panel")
            clump_summary.append({
                "dataset":        dataset,
                "chromosome":     chrom,
                "n_lead_snps":    n_clumps,
                "n_not_in_panel": n_missing,
            })
        else:
            print("failed — check log above")

print("\nClumping complete.")
print(pd.DataFrame(clump_summary).to_string(index=False))


── FinnGen ──
  Chr 1: clumping... 515 lead SNPs | 879 not in panel
  Chr 2: clumping... 563 lead SNPs | 885 not in panel
  Chr 4: clumping... 432 lead SNPs | 831 not in panel
  Chr 5: clumping... 761 lead SNPs | 1255 not in panel
  Chr 6: clumping... 379 lead SNPs | 699 not in panel
  Chr 7: clumping... 133 lead SNPs | 228 not in panel
  Chr 8: clumping... 478 lead SNPs | 1410 not in panel
  Chr 9: clumping... 174 lead SNPs | 189 not in panel
  Chr11: clumping... 632 lead SNPs | 1131 not in panel
  Chr12: clumping... 397 lead SNPs | 625 not in panel
  Chr13: clumping... 251 lead SNPs | 327 not in panel
  Chr15: clumping... 237 lead SNPs | 416 not in panel
  Chr16: clumping... 467 lead SNPs | 727 not in panel
  Chr17: clumping... 671 lead SNPs | 1101 not in panel
  Chr19: clumping... 726 lead SNPs | 1043 not in panel
  Chr20: clumping... 640 lead SNPs | 1139 not in panel
  Chr21: clumping... 269 lead SNPs | 379 not in panel

── GWAS Catalog ──

── HERMES ──
  Chr 1: clumping... 101 le

In [15]:
def parse_clumps_file(chrom, ds_tag, clump_raw_dir):
    clumps_file = clump_raw_dir / f"clumped_{ds_tag}_chr{chrom}.clumps"
    if not clumps_file.exists():
        return pd.DataFrame()
    df = pd.read_csv(clumps_file, sep=r"\s+")
    df["chromosome"] = chrom
    return df


def parse_missing_file(chrom, ds_tag, clump_raw_dir):
    missing_file = clump_raw_dir / f"clumped_{ds_tag}_chr{chrom}.clumps.missing_id"
    if not missing_file.exists():
        return set()
    with open(missing_file) as f:
        lines = f.read().strip().split("\n")
    return set(line.strip() for line in lines[1:] if line.strip())


all_clumps  = []
all_missing = set()

for dataset, info in dataset_inputs.items():
    ds_tag = dataset.lower().replace(" ", "_")

    for chrom in info["chromosomes"]:
        chrom_clumps  = parse_clumps_file(chrom, ds_tag, clump_raw_dir)
        chrom_missing = parse_missing_file(chrom, ds_tag, clump_raw_dir)

        if not chrom_clumps.empty:
            chrom_clumps["source_dataset"] = dataset
            all_clumps.append(chrom_clumps)

        all_missing.update(chrom_missing)

if all_clumps:
    clumps_df = pd.concat(all_clumps, ignore_index=True)
else:
    clumps_df = pd.DataFrame()

print(f"Total independent lead SNPs : {len(clumps_df):,}")
print(f"Total not in 1000G panel    : {len(all_missing):,}")

Total independent lead SNPs : 9,075
Total not in 1000G panel    : 13,617


In [16]:
if clumps_df.empty:
    print("No clumping results to process.")
else:
    hf_df["_key"] = hf_df["rsID"] + "||" + hf_df["source_dataset"]
    clumps_df["_key"] = clumps_df["ID"] + "||" + clumps_df["source_dataset"]
    lead_keys = set(clumps_df["_key"])

    hf_clumped = (
        hf_df[hf_df["_key"].isin(lead_keys)]
        .drop(columns="_key")
        .copy()
        .reset_index(drop=True)
    )

    hf_clumped = hf_clumped.sort_values("p_value").reset_index(drop=True)

    print(f"LD-clumped HF variants : {len(hf_clumped):,} rows")
    print(f"Unique lead rsIDs      : {hf_clumped['rsID'].nunique():,}")
    print(f"Unique genes           : {hf_clumped['gene'].nunique():,}")
    print(f"\nSource datasets:")
    print(hf_clumped["source_dataset"].value_counts().to_string())

LD-clumped HF variants : 9,650 rows
Unique lead rsIDs      : 8,706
Unique genes           : 51

Source datasets:
source_dataset
FinnGen    8231
HERMES     1419


In [17]:
final_columns = [
    "gene",
    "phenotype",
    "source_dataset",
    "rsID",
    "variant_id",
    "chromosome",
    "position",
    "genome_build",
    "effect_allele",
    "other_allele",
    "effect_size",
    "odds_ratio",
    "p_value",
    "allele_frequency",
    "location_relative_to_gene",
    "distance_from_gene",
    "functional_consequence",
    "gnomad_max_freq",
    "gnomad_max_freq_source",
    "gnomad_max_freq_population",
    "gnomad_priority",
]

# Keep only columns that actually exist (safety check)
final_columns = [c for c in final_columns if c in hf_clumped.columns]

hf_clumped_final = hf_clumped[final_columns].copy()

out_xlsx = out_dir / "hf_ld_clumped.xlsx"
out_csv  = out_dir / "hf_ld_clumped.csv"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    hf_clumped_final.to_excel(writer, sheet_name="LD Clumped HF Variants", index=False)

hf_clumped_final.to_csv(out_csv, sep=";", index=False)

print(f"Rows: {len(hf_clumped_final):,} | Columns: {len(hf_clumped_final.columns)}")

Rows: 9,650 | Columns: 21
